# 12 — Uncertainty, paired model comparison, and English retention

**Purpose:** Compare two complete model runs on exactly the same test items. Bootstrap confidence intervals describe uncertainty from limited test samples, not randomness from retraining. For robust conclusions, also use several training seeds.

**Start here:** The first code cell clones the public GitHub repository automatically when needed. Enable Internet in Kaggle/Colab.

In [ ]:
# 0. CLONE + ENVIRONMENT SETUP — RUN FIRST (Kaggle/Colab).
# Internet must be ON. The FIRST run may install libraries and STOP for a kernel restart.
# After restarting the kernel, rerun this cell; it will skip installation if versions match.
from pathlib import Path
import os, sys, subprocess, importlib.metadata as metadata

REPO_URL = "https://github.com/muhammad-omer-dev/nlp_research_proj.git"
PROJECT_FOLDER = "Urdu_Qwen_Research_Notebooks"
PULL_LATEST = False  # Set True after pushing changes to GitHub; requires a clean Git checkout.

if Path('/kaggle/working').is_dir():
    WORKSPACE = Path('/kaggle/working')
elif Path('/content').is_dir():
    WORKSPACE = Path('/content')
else:
    WORKSPACE = Path.cwd()
REPO_DIR = WORKSPACE / 'nlp_research_proj'
ROOT = REPO_DIR / PROJECT_FOLDER

if (REPO_DIR / '.git').is_dir():
    if PULL_LATEST:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    if REPO_DIR.exists():
        raise RuntimeError(f'{REPO_DIR} exists but is not a Git clone. Rename/remove it after backing up outputs.')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
if not (ROOT / 'requirements.txt').is_file() or not (ROOT / 'project_utils.py').is_file():
    raise FileNotFoundError(f'Expected complete project at {ROOT}. Check the GitHub folder layout.')

# This environment was selected for a stable Transformers 4 training API.
if not ((3, 10) <= sys.version_info[:2] <= (3, 12)):
    raise RuntimeError(f'Python {sys.version.split()[0]} not in tested support range 3.10–3.12.')
REQUIRED = {
    'transformers': '4.51.3',
    'peft': '0.15.2',
    'accelerate': '1.6.0',
    'datasets': '3.5.0',
    'tokenizers': '0.21.1',
    'huggingface-hub': '0.30.2',
    'safetensors': '0.5.3',
    'scikit-learn': '1.6.1',
    'pandas': '2.2.3',
    'numpy': '1.26.4',
    'matplotlib': '3.10.1',
    'psutil': '7.0.0',
    'ipywidgets': '8.1.6',
    'nbformat': '5.10.4',
    'pytest': '8.3.5',
}

def installed_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

mismatch = {name: (installed_version(name), version)
            for name, version in REQUIRED.items()
            if installed_version(name) != version}
torchao = installed_version('torchao')
if mismatch or torchao is not None:
    print('Installing the project-locked environment; CUDA-enabled PyTorch is left untouched.')
    if mismatch:
        print('Packages to align:', mismatch)
    if torchao is not None:
        print('Removing optional incompatible torchao:', torchao)
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'requirements.txt'),
                    '--upgrade-strategy', 'only-if-needed'], check=True)
    raise RuntimeError('Installation finished. RESTART THE KAGGLE/COLAB KERNEL, then rerun this FIRST cell. '
                       'Do not train before restarting.')

# No package installation needed: import the shared research code only now.
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from project_utils import *
folders()

# Catch future API drift before downloading a 0.5B model.
import inspect
from transformers import TrainingArguments, Trainer
sig = inspect.signature(TrainingArguments).parameters
needed = ['output_dir', 'group_by_length', 'warmup_ratio', 'eval_strategy']
missing = [name for name in needed if name not in sig]
if missing:
    raise RuntimeError(f'Transformers does not match our 4.51.3 training API. Missing: {missing}')
api = check_training_api()
print('Ready. Project:', ROOT)
print('Transformers:', api['transformers'], '| PEFT:', api['peft'])
print('Trainer token argument:', api['trainer_tokenizer_key'])
print('No installation needed; proceed to the next cell.')


## A. Paired bootstrap for LoRA versus full

Supply two complete prediction files from notebook 05 and make sure their metadata refer to the same dataset, split, and language. Invalid generated labels must remain wrong answers.

In [ ]:
TASK='sib200';LANG='ur';SPLIT='test'
FILE_A=None # example: sib200_lora_ur_n10_s42_r8_ur_test.csv
FILE_B=None # example: sib200_full_ur_n10_s42_ur_test.csv
N_BOOTSTRAP=1000
if FILE_A and FILE_B:
    pa,pb=[ROOT/'outputs/predictions'/name for name in [FILE_A,FILE_B]]
    ma,mb=read_json(pa.with_suffix('.meta.json')),read_json(pb.with_suffix('.meta.json'))
    validate_prediction_metadata(ma,TASK,LANG,SPLIT)
    validate_prediction_metadata(mb,TASK,LANG,SPLIT)
    assert not ma.get('subset') and not mb.get('subset')
    da=pd.DataFrame(read_csv(pa)).set_index('example_id').sort_index()
    db=pd.DataFrame(read_csv(pb)).set_index('example_id').sort_index()
    assert da.index.tolist()==db.index.tolist() and (da.reference==db.reference).all()
    labels=task_labels(TASK)
    y=da.reference.to_numpy();a=da.prediction.to_numpy();b=db.prediction.to_numpy()
    def score(ii,p): return f1_score(y[ii],p[ii],labels=labels,average='macro',zero_division=0)
    indices=np.arange(len(y));observed=score(indices,a)-score(indices,b)
    rng=np.random.default_rng(42)
    deltas=[score(ii:=rng.integers(0,len(y),len(y)),a)-score(ii,b) for _ in range(N_BOOTSTRAP)]
    low,high=np.percentile(deltas,[2.5,97.5])
    print('A minus B macro-F1:',round(observed,4))
    print('Paired bootstrap 95% interval:',round(low,4),round(high,4))
    print('TRAINING SEED VARIATION IS NOT INCLUDED: run 3 seeds to study training randomness.')
else:
    print('Set FILE_A and FILE_B to matching, complete prediction CSVs.')

## B. English performance retention

To investigate forgetting properly, evaluate the original English-trained anchor **before** Urdu training, and evaluate the Urdu-adapted LoRA and full models on the unchanged English test. The same labels and exact examples must be used.

In [ ]:
ENGLISH_BEFORE=None # e.g. outputs/metrics/<name>_metrics.json
ENGLISH_AFTER_LORA=None
ENGLISH_AFTER_FULL=None
if all([ENGLISH_BEFORE,ENGLISH_AFTER_LORA,ENGLISH_AFTER_FULL]):
    b,l,f=[read_json(ROOT/x) for x in [ENGLISH_BEFORE,ENGLISH_AFTER_LORA,ENGLISH_AFTER_FULL]]
    for x in [b,l,f]: assert x['task']=='sib200' and x['lang']=='en' and x['split']=='test'
    print('Before Urdu fine-tuning:',round(b['macro_f1'],4))
    print('English F1 change with Urdu LoRA:',round(l['macro_f1']-b['macro_f1'],4))
    print('English F1 change with Urdu full:',round(f['macro_f1']-b['macro_f1'],4))
else: print('Select all three English test metric JSON files.')

## C. Confusion matrix

This plot includes an additional INVALID category, so the classifier's formatting mistakes cannot disappear from the analysis.

In [ ]:
METRICS_FILE=None
if METRICS_FILE:
    record=read_json(ROOT/METRICS_FILE)
    cm=np.array(record['details']['confusion_matrix']); labels=record['details']['confusion_labels']
    plt.figure(figsize=(10,8));plt.imshow(cm,cmap='Blues')
    plt.xticks(range(len(labels)),labels,rotation=45,ha='right')
    plt.yticks(range(len(labels)),labels);plt.colorbar()
    plt.xlabel('Predicted');plt.ylabel('True');plt.title(record['run_id'])
    plt.tight_layout();plt.savefig(ROOT/'outputs/plots/confusion_matrix.png',dpi=170);plt.show()
else: print('Select a completed metrics JSON file.')